In [1]:
!pip3 install snscrape
from datetime import date
import snscrape.modules.twitter as sntwitter
import pandas as pd
import string
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [2]:
total_tweets = []
tot_Tweets = 1000

for count,tweet in enumerate(sntwitter.TwitterSearchScraper('from:UNICEF').get_items()):
    if count>tot_Tweets:
        break
    total_tweets.append([tweet.content.lower()])
    

tweets_df = pd.DataFrame(total_tweets, columns=['Tweets'])

tweets_df.to_csv("UNICEF_task1.csv", encoding='utf-8', index = False)

In [3]:
tweets_df = pd.read_csv('UNICEF_task1.csv')

In [4]:
tweets_df

,Tweets
0,"as climate change worsens, the decision at #co..."
1,"today is a day of action for children, by chil..."
2,every child and young person has a right to re...
3,"in a global poll, u-reporters said they tried ..."
4,from taking #climateaction to fighting against...
...,...
996,we're screening almost a million children a mo...
997,the war in ukraine is helping drive a global f...
998,"no matter the time, no matter the place, confl..."
999,the number of children suffering from hunger a...


In [5]:
def cleaner(text):
    text = re.sub('@[A-Za-z0-9_]+', '', text)
    text = re.sub('#[A-Za-z0-9_]+','', text)
    text = re.sub("-"," ",text)
    text  = ''.join([char for char in text if char not in string.punctuation])
    text = re.sub('[0-9]+', '', text)
    remo = re.compile(u'([\U00002600-\U000027BF])|([\U0001f300-\U0001f64F])|([\U0001f680-\U0001f6FF])|([\U0001F1E0-\U0001F1FF])|([\U0001F300-\U0001F5FF])|([\U0001F600-\U0001F64F])|([\U0001F680-\U0001F6FF])|([\U0001F700-\U0001F77F])|([\U0001F780-\U0001F7FF])|([\U0001F800-\U0001F8FF])|([\U0001F900-\U0001F9FF])|([\U0001FA00-\U0001FA6F])|([\U0001FA70-\U0001FAFF])|([\U00002702-\U000027B0])|([\U000024C2-\U0001F251]+)')
    text = remo.sub(r'', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub('RT[\s]+','',text)
    text = re.sub('https?:\/\/\S+', '', text) 
    text = re.sub("’", "", text)
    text = re.sub("–", "", text)
    text = re.sub("“","",text)
    text = re.sub("”", "", text)
    text = re.sub("°","",text)
    text = re.sub('[()!?]', ' ', text)
    text = re.sub('\[."*?\]',' ', text)
    text = re.sub('\n',' ',text)
    text = re.sub(r'(@|https?)\S+|#', '', text)
    text = re.sub('/[^\u1F600-\u1F6FF\s]/i', '', text)
    return text



tweets_df['newtweets'] = tweets_df['Tweets'].apply(cleaner)


In [6]:
tweets_df

,Tweets,newtweets
0,"as climate change worsens, the decision at #co...",as climate change worsens the decision at to ...
1,"today is a day of action for children, by chil...",today is a day of action for children by child...
2,every child and young person has a right to re...,every child and young person has a right to re...
3,"in a global poll, u-reporters said they tried ...",in a global poll u reporters said they tried h...
4,from taking #climateaction to fighting against...,from taking to fighting against gender based ...
...,...,...
996,we're screening almost a million children a mo...,were screening almost a million children a mon...
997,the war in ukraine is helping drive a global f...,the war in ukraine is helping drive a global f...
998,"no matter the time, no matter the place, confl...",no matter the time no matter the place conflic...
999,the number of children suffering from hunger a...,the number of children suffering from hunger a...


In [7]:
del tweets_df['Tweets']

tweets_df

,newtweets
0,as climate change worsens the decision at to ...
1,today is a day of action for children by child...
2,every child and young person has a right to re...
3,in a global poll u reporters said they tried h...
4,from taking to fighting against gender based ...
...,...
996,were screening almost a million children a mon...
997,the war in ukraine is helping drive a global f...
998,no matter the time no matter the place conflic...
999,the number of children suffering from hunger a...


In [8]:
tweets_df.to_csv("UNICEF_task2.csv", encoding='utf-8', index = False)

In [9]:
tweets_df = pd.read_csv('UNICEF_task2.csv')

In [10]:
tweets = tweets_df['newtweets']

X_train, X_test = train_test_split(tweets, random_state=42,test_size=0.2, shuffle=True)

In [11]:
print(X_train)

29     children and young people are taking  everywhe...
535    the learning crisis is a global challenge but ...
695    how many miles would you walk for your baby   ...
557    i like to read very much  for  year old fatima...
836    a three day old bundle of joy  baby mohammad w...
                             ...                        
106    flooding in asia  droughts in africa   wildfir...
270    its time to break the stigma around mental hea...
860    this is what were doing to prevent child hunge...
435    unicef goodwill ambassador  on the power of pl...
102    we asked unicef youth advocates what  means to...
Name: newtweets, Length: 800, dtype: object


In [12]:
def make_vocab(X_train):

  tweets = []
  for words in X_train:
      tweets.append(words)

  complete = []
  for line in tweets:
      word = line.split()
      for w in word:
          complete.append(w)

  vocab = list(set(complete)) #all the unique words in all the tweets

  return vocab

In [17]:
vocab = make_vocab(X_train)
print(len(vocab))

2701


In [14]:
def bag_of_words_maker(tweet):
  words = tweet.split()

  bag_o_words = [1] * len(vocab) #laplace smoothing

  for ind, word in enumerate(vocab):
    if(word in words):
      bag_o_words[ind] += 1
  
  return bag_o_words

In [15]:
counter = 1

for words in X_train:
  if(counter<=10):
    b_o_g = bag_of_words_maker(words)
    print(f"{counter}.  {b_o_g}")
    counter +=1
  else:
    break

1.  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [16]:
counter = 1

for words in X_test:
  if(counter<=10):
    b_o_g = bag_of_words_maker(words)
    print(f"{counter}.  {b_o_g}")
    counter +=1
  else:
    break

1.  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,